In [ ]:
import sys
sys.path.append("/Users/shikim/pynta_local/pynta/pynta")

# Preprocessing notebook

## Preprocessing notebook will calculate: 
1. notes for choice of pseudopotentials
2. Lattice parameter optimization and analysis
3. High surface index custom slab build
4. Energy convergence test
5. k-points optimization


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from ase.io import read
from ase.constraints import FixAtoms
from ase.visualize import view
from ase.build import surface

from pynta.calculator import get_lattice_parameters
from pynta.utils import name_to_ase_software

from pynta.preprocessing import *


### Notes on Pseudopotentials:

We can choose from various pseudopotential libraries. Choice of pseudopotential depends on the problem we are investigating, *e.g.,* if there is a heavy element present in our system and we are interested in the spin-orbit coupling effects, we should choose a full relativistic pseudopotential. We need to be careful whether our chosen pseudopotential correctly reproduces physical properties. Various pseudopotential libraries:

- https://www.quantum-espresso.org/pseudopotentials
- https://www.materialscloud.org/discover/sssp/table/efficiency
- http://www.pseudo-dojo.org
- https://www.physics.rutgers.edu/gbrv/
- https://nninc.cnf.cornell.edu
- http://www.quantum-simulation.org/potentials/
- BLYP pseudopotentials (https://pseudopotentials.quantum-espresso.org/legacy_tables/hartwigesen-goedecker-hutter-pp) 
- SCAN pseudopotentials (https://yaoyi92.github.io/scan-tm-pseudopotentials.html)


In [ ]:
#calculator
prep = Prep(
    metal="Pd",
    surface_type="fcc111",
    a0=3.89,
    software="Espresso",
    fmax=0.05,
    vacuum=10,
    software_kwargs={ # this keyword is to run convergence test
        "kpts": (3, 3, 1),
        "ecutwfc": 40,
        "ecutrho": 160,
        "occupations": "smearing",
        "smearing": "marzari-vanderbilt",
        "degauss": 0.01,
        "input_dft":"BEEF-VDW"
        "nosym": True,
        "conv_thr": 1e-6,
        "pseudopotentials": {
            "Pd": "pd_pbesol_v1.4.uspp.F.UPF",
            "H": "H.pbe-kjpaw_psl.1.0.0.UPF",
        },
    },
    lattice_opt_software_kwargs={ # this keyword is specifically for lattice constant optimization
            'kpts': (25,25,25), # something large
            'ecutwfc': 70,  # you can change this 
            'degauss':0.02,
            'input_dft':'BEEF-VDW', 
            'mixing_mode': 'plain'}, 
)

### 1. Lattice parameter optimization:


If your slab is not optimized, it is NOT recommended to use for Pynta production run. Please optimize your slab before running Pynta

*Preprocessing can only run with surfaces that ASE can build*

In [ ]:
# Calculate lattice constant
a_opt, outavals, Evals = prep.calculate_lattice_parameters(return_scan=True)

In [ ]:
# outavals, Evals already obtained via calculate_lattice_parameters()
a_interp, a_final = fit_lattice_constant_from_scan(
    outavals,
    Evals,
    title="Optimization of lattice constant with QE (cube cell)",
)


### 2. Build/load custom surface:

In [ ]:
# Create conventional slab
slab = prep.generate_slab()

In [ ]:
# Create custom slab
name = 'pd332.xyz'
n_layers = 10
slab = surface('Pd',(3,3,2),n_layers)
slab.center(vacuum=10, axis=2)

#visualize slab
write(name, slab)
view(slab, viewer='x3d')

In [ ]:
# Optimize custom slab
slab = prep.optimize_slab(
    a=3.89,
    c=None,
    slab_path=None,
    out_path="slab.xyz"
)

view(slab, viewer='x3d')


### 3. Cutoff Energy convergence test: 

We can calculate the total energy of the system for various values of energy cutoff values. 
Converged energy cutoff value with peudopotentials of choice can construct more accurate DFT input for Pynta. 

In [ ]:
ecut_values = range(30, 71, 10)  # Ry
kmesh = [(3, 3, 1)]

ecut_results = prep.run_convergence_adsorbate(
    ecut_values=ecut_values,
    kmesh_values=kmesh,
    adsorbate=None
) 

ecut_vals = [r[0] for r in ecut_results]
ecut_energies = [r[2] for r in ecut_results]


In [ ]:
plt.figure()
plt.plot(ecut_vals, ecut_energies, marker="o")
plt.xlabel("Energy cutoff (Ry)")
plt.ylabel("Total energy (eV)")
plt.title("Cutoff energy convergence")
plt.grid(True)
plt.show()

### 4. k-point convergence test:

With the optimized slab, appropriate k-points should be calculated to execute Pynta efficiently. 

Lowest total energy is estimated from lattice parameter optimization. We use the lowest potential energy to estimate k-points.

Tip: If you have already obtained satisfactory convergence with a (relatively) sparse k-point grid, there is no motivation to go for a denser grid. It makes calculations more expensive. 

In [ ]:
kpts_list = [(1, 1, 1), (2, 2, 1), (3, 3, 1), (4, 4, 1)]

kpt_results = prep.run_convergence_adsorbate(
    ecut_values=[40],   # fixed cutoff
    kmesh_values=kpts_list,
    adsorbate=None
)

kpt_labels = [str(r[1]) for r in kpt_results]
kpt_energies = [r[2] for r in kpt_results]


In [ ]:
plt.figure()
plt.plot(kpt_labels, kpt_energies, marker="o")
plt.xlabel("k-point mesh")
plt.ylabel("Total energy (eV)")
plt.title("k-point convergence")
plt.grid(True)
plt.show()
